# Task 6 — Auto-Correction

### Introduction
Spelling correction maps a misspelled word to the most likely correct target word.

This relies on two components:
1. **Edit Distance**: The minimum operations (insertions, deletions, substitutions, transpositions) required to convert one string into another. We implement the **Levenshtein Distance** algorithm using Dynamic Programming.
2. **Probability Ranking**: A candidate list is generated from edit-distance metrics. We filter candidates using a lexicon and rank them based on corpus frequency. The candidate with the highest frequency in natural language is chosen.

### Core Use Cases
- **Search Query Cleaning**: Fixing user typing errors in search input fields to return correct results.
- **Word Editors**: Powering inline spell-check suggestions in word processors.
- **Data Standardization**: Cleaning noisy web-scraped inputs or database entries containing spelling inconsistencies.



### Step 1 — Levenshtein Edit Distance from Scratch


In [1]:
import numpy as np
import pandas as pd

def edit_distance(word1, word2):
    m = len(word1)
    n = len(word2)
    dp = np.zeros((m + 1, n + 1), dtype=int)
    
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
        
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i-1] == word2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j],   # Deletion
                    dp[i][j-1],   # Insertion
                    dp[i-1][j-1]  # Substitution
                )
    return dp

# Test
w1, w2 = "speling", "spelling"
dp_matrix = edit_distance(w1, w2)
print(f"Edit Distance: {dp_matrix[-1][-1]}")



Edit Distance: 1


### Step 2 — Edit-Distance-1 Candidate Generator & Dictionary Builder


In [2]:
def edit_distance_1_candidates(word):
    letters = 'abcdefghijklmnopqrstuvwxyz'
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    
    deletes = [L + R[1:] for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R)>1]
    replaces = [L + c + R[1:] for L, R in splits if R for c in letters]
    inserts = [L + c + R for L, R in splits for c in letters]
    
    return set(deletes + transposes + replaces + inserts)

import nltk
from nltk.corpus import words
from collections import Counter

nltk.download('words', quiet=True)
nltk.download('webtext', quiet=True)
from nltk.corpus import webtext

word_list = words.words()
corpus_words = [w.lower() for w in webtext.words() if w.isalpha()]
word_freqs = Counter(corpus_words)

for w in word_list:
    w_low = w.lower()
    if w_low not in word_freqs:
        word_freqs[w_low] = 1

def correct_word(word):
    if word.lower() in word_freqs and word_freqs[word.lower()] > 1:
        return word
        
    candidates = edit_distance_1_candidates(word.lower())
    valid_candidates = [w for w in candidates if w in word_freqs]
    
    if not valid_candidates:
        return word
        
    best_candidate = max(valid_candidates, key=lambda w: word_freqs[w])
    return best_candidate



### Step 3 — User-Defined Interactive Evaluation Function
Enter a misspelled word below to query the candidate generator and return the suggested correction.


In [3]:
def evaluate_spelling_corrector(word):
    """
    User-defined evaluation function. Auto-corrects word input and prints details.
    """
    clean_word = word.strip().lower()
    corrected = correct_word(clean_word)
    
    print(f"Spelled Input: '{word}'")
    if clean_word == corrected:
        print("Status: Spelling is already correct.")
    else:
        print(f"Suggested Correction: '{corrected}'")
        
        # Calculate Levenshtein DP Matrix to show steps
        dp = edit_distance(clean_word, corrected)
        print(f"Levenshtein Edit Distance: {dp[-1][-1]}")

# Test custom inputs
evaluate_spelling_corrector("speling")
evaluate_spelling_corrector("computar")



Spelled Input: 'speling'
Suggested Correction: 'spelling'
Levenshtein Edit Distance: 1
Spelled Input: 'computar'
Suggested Correction: 'computer'
Levenshtein Edit Distance: 1
